In [1]:
import pandas as pd
import os

# Переходим в папку проекта
os.chdir(r'C:\Users\Lecoo\Documents\Projects\china_import_analysis')

# Загружаем все CSV-файлы из data/raw
raw_folder = 'data/raw'
csv_files = [f for f in os.listdir(raw_folder) if f.endswith('.csv')]

# Читаем и склеиваем все файлы
list_of_dataframes = []
for file in csv_files:
    df_temp = pd.read_csv(os.path.join(raw_folder, file), encoding='windows-1251')
    df_temp['source_file'] = file
    list_of_dataframes.append(df_temp)

df_all = pd.concat(list_of_dataframes, ignore_index=True)


In [2]:
# === Создаем справочник техники на русском языке
product_mapping = {
    84291110: {'product': 'бульдозеры', 'details': 'бульдозеры до 235 кВт'},
    84291190: {'product': 'бульдозеры', 'details': 'бульдозеры больше 235 кВт'},
    84292010: {'product': 'грейдеры', 'details': 'грейдеры до 235 кВт'},
    84292090: {'product': 'грейдеры', 'details': 'грейдеры больше 235 кВт'},
    84295100: {'product': 'фронтальные погрузчики', 'details': 'фронтальные погрузчики'},
    84295211: {'product': 'колесные экскаваторы', 'details': 'колесные экскаваторы'},
    84295212: {'product': 'гусеничные экскаваторы', 'details': 'гусеничные экскаваторы'},
    84295900: {'product': 'экскаваторы-погрузчики', 'details': 'экскаваторы-погрузчики'}
}

mapping_df = pd.DataFrame.from_dict(product_mapping, orient='index').reset_index()
mapping_df.columns = ['Commodity code', 'Product', 'Details']

In [3]:
# === Очистка
df_all = df_all.merge(mapping_df, on='Commodity code', how='left')
# Убираем продукты, которые не нужны для проекта
df_all = df_all.dropna(subset=['Product']).copy()

In [4]:
# === Обработка даты
df_all['date'] = pd.to_datetime(df_all['Date of data'].astype(str) + '01', format='%Y%m%d')

df_all['year'] = df_all['date'].dt.year
df_all['month'] = df_all['date'].dt.month
df_all['month_name'] = df_all['date'].dt.month_name()
df_all['quarter'] = df_all['date'].dt.quarter

df_all = df_all.drop('Date of data', axis=1)
df_all.columns = df_all.columns.str.lower()

In [5]:
df_all.tail()

,commodity code,commodity,trading partner code,trading partner,customs regime code,customs regime,locations of importers and exporters code,locations of importers and exporters,quantity,unit,...,us dollar,unnamed: 14,source_file,product,details,date,year,month,month_name,quarter
11119,84295900,Self-propelled excavating machinery nes,344,Russia,10,Ordinary Trade,45,Guangxi Zhuang Autonomous Region,2,Number of item,...,"23,998",NaN,2026.csv,экскаваторы-погрузчики,экскаваторы-погрузчики,2026-06-01,2026,6,June,2
11120,84295900,Self-propelled excavating machinery nes,344,Russia,15,Processing with Imported Materials,37,Shandong Province,22,Number of item,...,"1,219,575",NaN,2026.csv,экскаваторы-погрузчики,экскаваторы-погрузчики,2026-06-01,2026,6,June,2
11121,84295900,Self-propelled excavating machinery nes,344,Russia,19,Border Trade,15,Inner Mongolia Autonomous Region,1,Number of item,...,"41,327",NaN,2026.csv,экскаваторы-погрузчики,экскаваторы-погрузчики,2026-06-01,2026,6,June,2
11122,84295900,Self-propelled excavating machinery nes,344,Russia,19,Border Trade,23,Heilongjiang Province,2,Number of item,...,"28,690",NaN,2026.csv,экскаваторы-погрузчики,экскаваторы-погрузчики,2026-06-01,2026,6,June,2
11123,84295900,Self-propelled excavating machinery nes,344,Russia,34,Entrepot Trade by Bonded Area,61,Shaanxi Province,2,Number of item,...,"29,405",NaN,2026.csv,экскаваторы-погрузчики,экскаваторы-погрузчики,2026-06-01,2026,6,June,2


In [6]:
# === Ячейка 5: Финальная проверка перед загрузкой ===
print('Форма:', df_all.shape)
print('\nТипы данных:')
print(df_all.dtypes)
print('\nПропуски по колонкам:')
print(df_all.isna().sum())
print('\nУникальные продукты:')
print(df_all['product'].value_counts())
print('\nДиапазон дат:', df_all['date'].min(), '—', df_all['date'].max())

Форма: (6684, 22)

Типы данных:
commodity code                                        int64
commodity                                            object
trading partner code                                  int64
trading partner                                      object
customs regime code                                   int64
customs regime                                       object
locations of importers and exporters code             int64
locations of importers and exporters                 object
quantity                                              int64
unit                                                 object
supplementary quantity                                int64
supplementary unit                                   object
us dollar                                            object
unnamed: 14                                         float64
source_file                                          object
product                                              object
details 

In [9]:
import sqlite3
import os

# Путь к файлу (в той же папке data)
db_path = 'data/analytics.sqlite'

# Удаляем старый файл, если есть
if os.path.exists(db_path):
    os.remove(db_path)

# Создаём подключение и пишем таблицу
conn = sqlite3.connect(db_path)
df_all.to_sql('china_import', conn, if_exists='replace', index=False)
conn.close()

print(f'SQLite база создана: {os.path.abspath(db_path)}')
print(f'Строк: {len(df_all)}')

SQLite база создана: C:\Users\Lecoo\Documents\Projects\china_import_analysis\data\analytics.sqlite
Строк: 6684
